## 🤖 LangChain + LangGraph Agent 完整示例

**特性一览：**
- **LLM**: DeepSeek (通过 OpenAI 兼容接口)
- **短期记忆**: LangGraph 状态中的消息列表 (`add_messages`)，自动保持对话上下文
- **长期记忆**: `SqliteSaver` 持久化 checkpoint 到磁盘，重启后仍可恢复
- **Function Call (工具调用)**:
  - `run_python_code` — 执行任意 Python 代码片段并返回结果
  - `get_weather` — 模拟天气查询
  - `calculate` — 数学计算器

In [5]:
# ==============================================================================
# LangChain + LangGraph Agent 示例
# 功能：DeepSeek 模型 + 短期记忆 + SQLite 持久化记忆 + Function Call（Python 执行工具）
# ==============================================================================

import io
import sys
import sqlite3
from typing import Annotated, TypedDict, Sequence

# ----- LangChain 核心组件 -----
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, ToolMessage
from langchain_core.tools import tool
from langchain_openai import ChatOpenAI

# ----- LangGraph 图编排组件 -----
from langgraph.graph import StateGraph, START, END
from langgraph.graph.message import add_messages
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.checkpoint.sqlite import SqliteSaver

print("✅ 所有依赖导入成功！")

✅ 所有依赖导入成功！


In [6]:
# ==============================================================================
# 第一部分：定义工具 (Tools) — Function Call 的核心
# ==============================================================================

@tool
def run_python_code(code: str) -> str:
    """
    执行一段 Python 代码并返回标准输出结果。
    支持数学运算、字符串处理、列表操作、数据分析等。
    代码中可以使用 print() 来输出结果。
    """
    old_stdout = sys.stdout
    redirected_output = io.StringIO()
    sys.stdout = redirected_output
    try:
        # 创建一个安全的执行环境
        exec_globals = {"__builtins__": __builtins__}
        exec(code, exec_globals)
        output = redirected_output.getvalue()
        return output.strip() if output.strip() else "代码执行成功（无输出）"
    except Exception as e:
        return f"执行出错: {type(e).__name__}: {e}"
    finally:
        sys.stdout = old_stdout


@tool
def get_weather(location: str) -> str:
    """获取指定城市的当前天气状况。"""
    weather_db = {
        "北京": "☀️ 晴天, 气温 22°C, 湿度 35%",
        "上海": "🌧️ 小雨, 气温 18°C, 湿度 78%",
        "广州": "⛅ 多云, 气温 28°C, 湿度 65%",
        "深圳": "🌤️ 晴间多云, 气温 27°C, 湿度 60%",
        "成都": "🌫️ 阴天, 气温 16°C, 湿度 80%",
    }
    for city, weather in weather_db.items():
        if city in location:
            return f"{city}: {weather}"
    return f"{location}: 暂无天气数据，建议查看天气预报网站。"


@tool
def calculate(expression: str) -> str:
    """一个安全的数学计算器，可以计算数学表达式，如: 2**10, 3.14*5**2, 100/3 等。"""
    import math
    try:
        # 只允许数学相关的内建函数
        allowed = {k: v for k, v in math.__dict__.items() if not k.startswith('_')}
        allowed.update({"abs": abs, "round": round, "min": min, "max": max, "sum": sum})
        result = eval(expression, {"__builtins__": {}}, allowed)
        return f"计算结果: {expression} = {result}"
    except Exception as e:
        return f"计算失败: {e}"


# 注册所有工具
tools = [run_python_code, get_weather, calculate]
print(f"✅ 已注册 {len(tools)} 个工具: {[t.name for t in tools]}")

✅ 已注册 3 个工具: ['run_python_code', 'get_weather', 'calculate']


In [7]:
# ==============================================================================
# 第二部分：构建 LangGraph Agent
# - 短期记忆：通过 AgentState.messages (add_messages) 自动维护对话上下文
# - 长期记忆：通过 SqliteSaver 将 checkpoint 持久化到 SQLite 数据库
# ==============================================================================

# --- 1. 定义 Agent 状态 ---
class AgentState(TypedDict):
    # add_messages 注解：每次返回的消息会 **追加** 到列表中，而不是覆盖
    # 这就是「短期记忆」的核心——完整的对话历史在图的每一步中自动维护
    messages: Annotated[Sequence[BaseMessage], add_messages]


# --- 2. 初始化 DeepSeek 模型 ---
DEEPSEEK_API_KEY = "sk-bdba15c4203448edbae37f85faab5c27"

llm = ChatOpenAI(
    api_key=DEEPSEEK_API_KEY,
    base_url="https://api.deepseek.com",
    model="deepseek-chat",
    temperature=0,
)

# 绑定工具：将 Python 函数自动转为 OpenAI 兼容的 function schema
llm_with_tools = llm.bind_tools(tools)


# --- 3. 定义节点逻辑 ---
def agent_node(state: AgentState):
    """
    Agent 推理节点：接收完整对话历史（短期记忆），调用 LLM 进行推理。
    LLM 可能直接回复文本，也可能返回 tool_calls 请求调用工具。
    """
    response = llm_with_tools.invoke(state["messages"])
    return {"messages": [response]}


# 工具执行节点（LangGraph 预置）：自动解析 tool_calls 并执行对应函数
tool_node = ToolNode(tools)


# --- 4. 构建 StateGraph ---
workflow = StateGraph(AgentState)

# 添加节点
workflow.add_node("agent", agent_node)    # 推理节点
workflow.add_node("tools", tool_node)     # 工具执行节点

# 添加边
workflow.add_edge(START, "agent")         # 入口 → agent
workflow.add_conditional_edges(
    "agent",
    tools_condition,                       # 若 LLM 返回 tool_calls → tools 节点；否则 → END
)
workflow.add_edge("tools", "agent")       # 工具执行完 → 回到 agent 继续推理


# --- 5. 编译图 + 接入 SQLite 持久化记忆 ---
# SqliteSaver 是「长期记忆」：每一轮对话的完整状态都会写入磁盘
# 即使程序重启，也可通过 thread_id 恢复之前的对话上下文
conn = sqlite3.connect("memory_demo.sqlite", check_same_thread=False)
memory_saver = SqliteSaver(conn)

app = workflow.compile(checkpointer=memory_saver)

print("✅ Agent 构建完成！")
print("   📝 短期记忆: AgentState.messages (add_messages 自动追加对话历史)")
print("   💾 长期记忆: SqliteSaver → memory_demo.sqlite")
print("   🔧 工具: run_python_code, get_weather, calculate")

✅ Agent 构建完成！
   📝 短期记忆: AgentState.messages (add_messages 自动追加对话历史)
   💾 长期记忆: SqliteSaver → memory_demo.sqlite
   🔧 工具: run_python_code, get_weather, calculate


In [8]:
# ==============================================================================
# 第三部分：定义辅助函数 & 运行 Agent
# ==============================================================================

def chat(user_input: str, thread_id: str = "notebook_demo_001"):
    """
    向 Agent 发送一条消息并打印完整的推理过程。
    
    Args:
        user_input: 用户输入的消息
        thread_id: 会话线程 ID，同一 thread_id 共享记忆
    """
    config = {"configurable": {"thread_id": thread_id}}
    
    print(f"\n{'='*60}")
    print(f"👤 [用户]: {user_input}")
    print(f"{'='*60}")
    
    # 使用 stream 模式，可以观察图中每个节点的执行过程
    events = app.stream(
        {"messages": [HumanMessage(content=user_input)]},
        config,
        stream_mode="values",
    )
    
    print("\n--- 🔄 Agent 推理过程 ---")
    for event in events:
        latest_msg = event["messages"][-1]
        
        if isinstance(latest_msg, HumanMessage):
            continue
        elif isinstance(latest_msg, AIMessage):
            if latest_msg.tool_calls:
                for tc in latest_msg.tool_calls:
                    print(f"  🤖 [Agent 决策]: 调用工具 → {tc['name']}")
                    # 对于 run_python_code，显示要执行的代码
                    if tc['name'] == 'run_python_code':
                        code = tc['args'].get('code', '')
                        print(f"     📋 执行代码:\n     {code}")
                    else:
                        print(f"     📋 参数: {tc['args']}")
        elif isinstance(latest_msg, ToolMessage):
            print(f"  ⚙️  [工具结果] {latest_msg.name}: {latest_msg.content}")
    
    # 获取最终回复
    final_state = app.get_state(config)
    final_answer = final_state.values["messages"][-1].content
    
    print(f"\n--- 💬 最终回复 ---")
    print(f"🤖 [AI助手]: {final_answer}")
    print(f"{'='*60}\n")
    
    return final_answer

In [9]:
# ==============================================================================
# 演示 1：Function Call — 调用 Python 执行工具
# Agent 会自动判断需要执行 Python 代码，并通过 run_python_code 工具完成
# ==============================================================================

chat("请用Python帮我计算斐波那契数列的前20个数，并打印出来")


👤 [用户]: 请用Python帮我计算斐波那契数列的前20个数，并打印出来

--- 🔄 Agent 推理过程 ---
  🤖 [Agent 决策]: 调用工具 → run_python_code
     📋 执行代码:
     def fibonacci(n):
    """计算斐波那契数列的前n个数"""
    fib_sequence = []
    a, b = 0, 1
    for i in range(n):
        fib_sequence.append(a)
        a, b = b, a + b
    return fib_sequence

# 计算前20个斐波那契数
fib_numbers = fibonacci(20)

print("斐波那契数列的前20个数：")
for i, num in enumerate(fib_numbers, 1):
    print(f"第{i:2d}项: {num:7d}")

print(f"\n前20个斐波那契数列表：")
print(fib_numbers)
  ⚙️  [工具结果] run_python_code: 斐波那契数列的前20个数：
第 1项:       0
第 2项:       1
第 3项:       1
第 4项:       2
第 5项:       3
第 6项:       5
第 7项:       8
第 8项:      13
第 9项:      21
第10项:      34
第11项:      55
第12项:      89
第13项:     144
第14项:     233
第15项:     377
第16项:     610
第17项:     987
第18项:    1597
第19项:    2584
第20项:    4181

前20个斐波那契数列表：
[0, 1, 1, 2, 3, 5, 8, 13, 21, 34, 55, 89, 144, 233, 377, 610, 987, 1597, 2584, 4181]

--- 💬 最终回复 ---
🤖 [AI助手]: 斐波那契数列的前20个数已经计算出来了！这个数列的特点是：从第3项开始，每一项都等于前两项之和。

**斐波那契数列前20项总结

'斐波那契数列的前20个数已经计算出来了！这个数列的特点是：从第3项开始，每一项都等于前两项之和。\n\n**斐波那契数列前20项总结：**\n- 第1项：0\n- 第2项：1\n- 第3项：1\n- 第4项：2\n- 第5项：3\n- 第6项：5\n- 第7项：8\n- 第8项：13\n- 第9项：21\n- 第10项：34\n- 第11项：55\n- 第12项：89\n- 第13项：144\n- 第14项：233\n- 第15项：377\n- 第16项：610\n- 第17项：987\n- 第18项：1597\n- 第19项：2584\n- 第20项：4181\n\n可以看到，随着项数的增加，数值增长得非常快。这个数列在数学、自然界和计算机科学中都有广泛的应用。'

In [10]:
# ==============================================================================
# 演示 2：Function Call — 多工具协同
# Agent 可能同时调用天气查询和计算器
# ==============================================================================

chat("北京今天天气怎么样？另外帮我算一下 2 的 32 次方是多少")


👤 [用户]: 北京今天天气怎么样？另外帮我算一下 2 的 32 次方是多少

--- 🔄 Agent 推理过程 ---
  🤖 [Agent 决策]: 调用工具 → get_weather
     📋 参数: {'location': '北京'}
  🤖 [Agent 决策]: 调用工具 → calculate
     📋 参数: {'expression': '2**32'}
  ⚙️  [工具结果] calculate: 计算结果: 2**32 = 4294967296

--- 💬 最终回复 ---
🤖 [AI助手]: **北京今天的天气：**
- 天气状况：☀️ 晴天
- 气温：22°C
- 湿度：35%

今天北京天气很好，是晴天，温度22°C比较舒适，湿度35%也比较适宜。

**2的32次方计算结果：**
2³² = 4,294,967,296

这个数字在计算机科学中很重要，因为：
1. 它是32位无符号整数的最大值
2. 在IPv4地址中，总共有2³²个可能的地址（约43亿个）
3. 在内存寻址中，32位系统最多可以寻址4GB内存



'**北京今天的天气：**\n- 天气状况：☀️ 晴天\n- 气温：22°C\n- 湿度：35%\n\n今天北京天气很好，是晴天，温度22°C比较舒适，湿度35%也比较适宜。\n\n**2的32次方计算结果：**\n2³² = 4,294,967,296\n\n这个数字在计算机科学中很重要，因为：\n1. 它是32位无符号整数的最大值\n2. 在IPv4地址中，总共有2³²个可能的地址（约43亿个）\n3. 在内存寻址中，32位系统最多可以寻址4GB内存'

In [ ]:
# ==============================================================================
# 演示 3：短期记忆 — 验证上下文保持能力
# 同一 thread_id 下，Agent 能记住之前的对话内容
# ==============================================================================

# 这里 Agent 应该能回忆起上面问过的斐波那契和天气问题
chat("我刚才问了你哪些问题？请帮我总结一下")

In [ ]:
# ==============================================================================
# 演示 4：Function Call — Python 数据分析示例
# 更复杂的 Python 代码执行场景
# ==============================================================================

chat("""请用Python完成以下任务：
1. 生成一个包含 1 到 100 的列表
2. 筛选出其中所有的质数
3. 计算这些质数的总和
4. 打印质数列表和总和""")

In [ ]:
# ==============================================================================
# 演示 5：长期记忆（SqliteSaver）— 验证持久化
# 通过读取已保存的状态来验证 SQLite 记忆是否生效
# ==============================================================================

# 查看当前会话的完整消息历史（从 SQLite 中恢复）
config = {"configurable": {"thread_id": "notebook_demo_001"}}
saved_state = app.get_state(config)

print("💾 从 SQLite 中恢复的对话历史：")
print(f"   总消息数: {len(saved_state.values['messages'])}")
print("\n📜 对话记录摘要:")
for i, msg in enumerate(saved_state.values["messages"]):
    role = type(msg).__name__.replace("Message", "")
    content_preview = msg.content[:80] + "..." if len(msg.content) > 80 else msg.content
    if content_preview:  # 跳过空内容的消息
        print(f"   [{i+1}] {role}: {content_preview}")

print("\n✅ SQLite 持久化记忆验证完成！")
print("   即使重启 Notebook，只要 memory_demo.sqlite 文件存在，")
print("   使用相同的 thread_id 就能恢复完整对话。")

In [ ]:
# ==============================================================================
# 演示 6：新会话 — 验证不同 thread_id 的记忆隔离
# ==============================================================================

# 使用不同的 thread_id，Agent 不会记得之前的对话
chat("你还记得我之前问过什么吗？", thread_id="new_session_002")